# 描述性统计 
描述性统计部分，主要统计Agent的分布，学习曲线等结果

## 导入库

In [1]:
import os
import re
import warnings
import polars as pl
import plotly  
import plotly.express as px
from plotly.subplots import make_subplots

## 超参数

In [2]:
TASK_ID_PREFIX = 'lstm_short'  # 任务id前缀
RESULTS_BASE_DIR = '/home/frank/files/programs/GraduationThesis/result' # 基本数据路径
SAVE_BASE_DIR = f'/home/frank/files/programs/GraduationThesis/empirical/{TASK_ID_PREFIX}' # 保存基本路径
SAVE = True # 是否保存数据

## 读取agent的meta，统计agent的能力分布  

In [3]:
# 列出task_id下的所有no
pattern = re.compile(r'\d{8}_\d{4}_' + TASK_ID_PREFIX + r'_[a-z0-9\-]+')
matching_task_ids = [task_id for task_id in os.listdir(RESULTS_BASE_DIR) if pattern.match(task_id)] # 匹配中缀 

# 获取其下所有node的路径
node_paths = [] # 所有该task_id中缀的node路径
for task_id in matching_task_ids:
    node_paths.extend(
        os.path.join(
            RESULTS_BASE_DIR, task_id, 
            node_path
        )
        for node_path in os.listdir(os.path.join(RESULTS_BASE_DIR, task_id))
    ) 

node_paths

['/home/frank/files/programs/GraduationThesis/result/20260215_1924_lstm_short_1231231232512-124n1232-23n123l-2312/node124',
 '/home/frank/files/programs/GraduationThesis/result/20260215_1924_lstm_short_1231231232512-124n1232-23n123l-2312/node125',
 '/home/frank/files/programs/GraduationThesis/result/20260215_1924_lstm_short_1231231232512-124n1232-23n123l-2312/node123',
 '/home/frank/files/programs/GraduationThesis/result/20260214_1924_lstm_short_66ab0230-9c11-4dc5-90e4-feb4b2c2fa57/node124',
 '/home/frank/files/programs/GraduationThesis/result/20260214_1924_lstm_short_66ab0230-9c11-4dc5-90e4-feb4b2c2fa57/node125',
 '/home/frank/files/programs/GraduationThesis/result/20260214_1924_lstm_short_66ab0230-9c11-4dc5-90e4-feb4b2c2fa57/node123']

读取所有node的meta.json，保存为lf  

需要对lf进行过滤： 

- 1.meta的顶层字段，都是一样的，仅需要第一行的内容就行  

- 2.meta的train_config字段，每一个agent都不同，需要单独展开，进行统计  

In [4]:
# 读取数据
lf_list = [pl.scan_ndjson(os.path.join(node_path, 'meta.json')) for node_path in node_paths]
lf_df = pl.concat(lf_list, rechunk=True)

1.顶层字段  
meta的顶层字段，是所有证券的共同字段，仅需要第一行的内容就行，但需要处理  
- 将task_id字段改为task_prefix，相同的task，prefix一致，但是uuid不同  
- 展开env_config  
- 展开performance_config  

In [5]:
meta_lf = lf_df.head(1)
meta_lf = meta_lf.with_columns(pl.lit(TASK_ID_PREFIX).alias('task_prefix'))
meta_lf = meta_lf.select(pl.all().exclude(['train_config','task_id']))
meta_lf = meta_lf.with_columns(pl.lit(1).alias('start_month')) # 添加一个start_month字段，为1 (和end_month字段对应)
meta_lf = meta_lf.with_columns(pl.col('env_config').struct.unnest()).select(pl.all().exclude(['env_config']))
meta_lf = meta_lf.with_columns(pl.col('performance_config').struct.unnest()).select(pl.all().exclude(['performance_config']))
meta_lf.head().collect()

start_year,N,stock_list,n,max_portfolios_num,factors_list,end_year,end_month,earliest_year_month,short_limit,task_prefix,start_month,sample_and_shuffle_seed,rl_end_year,save_model_every_n_steps,save_record_every_n_steps,save_performance_and_reward_every_n_steps,box_max,box_min,risk_free_rate,rolling_window,std_window,std_floor
i64,i64,list[str],i64,i64,list[str],i64,i64,list[i64],f64,str,i32,i64,i64,i64,i64,i64,i64,i64,f64,i64,i64,f64
2010,10,"[""000001"", ""000002"", … ""000010""]",1,252,"[""absacc"", ""acc"", … ""volumed""]",2020,12,"[1997, 1]",-0.3,"""lstm_short""",1,42,2015,2500,300,2500,1,-1,0.0,24,24,0.01


In [6]:
if SAVE:
    meta_lf.collect().write_parquet(os.path.join(SAVE_BASE_DIR, 'meta.parquet'))

2.train_config字段    
train_config字段，是每个agent的训练参数，随机采样而来，需要进行描述性统计   

在model_config 下有config字段，reinforcement_config下有config字段，这两个字段需要分别按照cate分类统计  


In [7]:
train_config = lf_df.select('train_config')
train_config = train_config.with_columns(pl.col('train_config').struct.unnest()).select(pl.all().exclude(['train_config']))

首先统计神经网络的config，保存为model_unique_config   
按照cate分类，对于每一个字段，统计其均值方差，绘制柱状图  

In [8]:
def stats(lf:pl.LazyFrame,field:str)->tuple[pl.DataFrame, plotly.graph_objects.Figure]:
    """
    对于df种的field字段，统计其均值方差等指标，绘制柱状图
    
    输入： 
    - lf: pl.LazyFrame 
    - field: str 需要统计的field字段  

    输出：
    - df: pl.DataFrame 统计后的数据(mean,std,median,modes,min,max)
    - fig: plotly.graph_objects.Figure 柱状图  
    """
    if field not in lf.collect_schema().names():
        warnings.warn(f'{field} not in lf')
        return 
    
    field_col = lf.select(field) # 获取field字段 
    field_stats = field_col.select(
        pl.col(field).mean().alias('mean'),
        pl.col(field).std().alias('std'),
        pl.col(field).median().alias('median'),
        pl.col(field).mode().alias('modes'),
        pl.col(field).min().alias('min'),
        pl.col(field).max().alias('max')
    )
    fig = px.histogram(
        lf.collect().to_pandas(),
        x=field,
        histnorm='probability density',
        title=f'{field} distribution (density)',
    )
    return field_stats.collect(), fig



In [9]:
model_config = train_config.select('model_config')
model_config = model_config.with_columns(pl.col('model_config').struct.unnest()).select(pl.all().exclude(['model_config']))  
model_unique_config = model_config.select(['cate','config']).with_columns(pl.col('config').struct.unnest()).select(pl.all().exclude(['config']))

# 筛选模型
mlp = model_unique_config.filter(pl.col('cate') == 0)
tcn = model_unique_config.filter(pl.col('cate') == 1)
lstm = model_unique_config.filter(pl.col('cate') == 2)

统计三个类型的模型，分别统计模型个数，各参数指标和柱状图  

In [10]:
for m in [0,1,2]:
    if m == 0:
        model_name = 'mlp'
        model = mlp 
    elif m == 1:
        model_name = 'tcn'
        model = tcn 
    elif m == 2:
        model_name = 'lstm'
        model = lstm 
    
    print(f'==================== {model_name} ====================')
    model_count = model.collect().shape[0]
    print(f'模型个数：{model_count}')
    if model_count == 0:
        continue
    print('模型参数统计')
    fields = [c for c in model.collect_schema().names() if c != 'cate']
    n = len(fields)
    if n > 0:
        n_cols = 2
        n_rows = (n + n_cols - 1) // n_cols
        fig_combined = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=fields, vertical_spacing=0.12, horizontal_spacing=0.08)
        for i, c in enumerate(fields):
            field_stats, fig = stats(model, c)
            print('------' + c + '------')
            print(field_stats)
            row, col = i // n_cols + 1, i % n_cols + 1
            for trace in fig.data:
                fig_combined.add_trace(trace, row=row, col=col)
        fig_combined.update_layout(height=300 * n_rows, title_text=f'{model_name} 各字段分布', showlegend=False)
        fig_combined.show()
        if SAVE:
            fig_combined.write_image(os.path.join(SAVE_BASE_DIR, f'{model_name}_各字段分布.png'))
    print('\n')


            

==================== mlp ====================
模型个数：0
==================== tcn ====================
模型个数：0
==================== lstm ====================
模型个数：6
模型参数统计
------hidden_size------
shape: (1, 6)
┌──────┬─────┬────────┬───────┬─────┬─────┐
│ mean ┆ std ┆ median ┆ modes ┆ min ┆ max │
│ ---  ┆ --- ┆ ---    ┆ ---   ┆ --- ┆ --- │
│ f64  ┆ f64 ┆ f64    ┆ i64   ┆ i64 ┆ i64 │
╞══════╪═════╪════════╪═══════╪═════╪═════╡
│ 60.0 ┆ 0.0 ┆ 60.0   ┆ 60    ┆ 60  ┆ 60  │
└──────┴─────┴────────┴───────┴─────┴─────┘
------num_layers------
shape: (1, 6)
┌──────┬─────┬────────┬───────┬─────┬─────┐
│ mean ┆ std ┆ median ┆ modes ┆ min ┆ max │
│ ---  ┆ --- ┆ ---    ┆ ---   ┆ --- ┆ --- │
│ f64  ┆ f64 ┆ f64    ┆ i64   ┆ i64 ┆ i64 │
╞══════╪═════╪════════╪═══════╪═════╪═════╡
│ 1.0  ┆ 0.0 ┆ 1.0    ┆ 1     ┆ 1   ┆ 1   │
└──────┴─────┴────────┴───────┴─────┴─────┘
------bidirectional------
shape: (1, 6)
┌──────┬─────┬────────┬───────┬──────┬──────┐
│ mean ┆ std ┆ median ┆ modes ┆ min  ┆ max  │
│ ---  ┆ -

In [11]:


# 展开env_config
lf_df = (
    lf_df
    .with_columns(
        pl.col('env_config').struct.field()
    )
)
lf_df.head().collect()




TypeError: ExprStructNameSpace.field() missing 1 required positional argument: 'name'